In [2]:
from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver


In [3]:
load_dotenv()

llm = ChatOpenAI(
    model = "gpt-4o-mini",
    temperature=0
)

In [4]:
class JokeState(TypedDict):
    topic : str
    joke: str
    explanation : str

In [5]:
def generate_joke(state: JokeState):
    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [6]:
def generate_explanation(state: JokeState):
    prompt = f'generate the explanation of this joke{state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation' , generate_explanation)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [8]:
config1 = {"configurable": {"thread_id":"1"}}
workflow.invoke({'topic':'Football'}, config=config1)

{'topic': 'Football',
 'joke': 'Why did the football team go to the bank?\n\nBecause they wanted to get their quarterback!',
 'explanation': 'This joke plays on a clever pun involving the word "quarterback." In American football, a quarterback is a key player who leads the team\'s offense and is responsible for throwing passes and making strategic plays. The humor arises from the double meaning of "quarterback." \n\nOn one hand, "quarterback" refers to the player on the football team. On the other hand, the joke suggests that the team went to the bank to "get" their quarterback as if they were withdrawing money. The play on words creates a humorous image of a football team treating their quarterback like a bank transaction, leading to the punchline. The unexpected twist of combining sports terminology with banking makes the joke amusing.'}

In [9]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Football', 'joke': 'Why did the football team go to the bank?\n\nBecause they wanted to get their quarterback!', 'explanation': 'This joke plays on a clever pun involving the word "quarterback." In American football, a quarterback is a key player who leads the team\'s offense and is responsible for throwing passes and making strategic plays. The humor arises from the double meaning of "quarterback." \n\nOn one hand, "quarterback" refers to the player on the football team. On the other hand, the joke suggests that the team went to the bank to "get" their quarterback as if they were withdrawing money. The play on words creates a humorous image of a football team treating their quarterback like a bank transaction, leading to the punchline. The unexpected twist of combining sports terminology with banking makes the joke amusing.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1e27a0-e759-6c1d-8002-c5916409c24a

In [10]:
config2 = {"configurable":{"thread_id":"2"}}
workflow.invoke({'topic':'cricket'},config=config2)

{'topic': 'cricket',
 'joke': 'Why did the cricket team bring a ladder to the match?\n\nBecause they heard the game was going to be a "high" scoring one!',
 'explanation': 'This joke plays on a pun involving the word "high." In cricket, a "high-scoring" game refers to a match where teams score a lot of runs, indicating an exciting and competitive game. However, the joke takes this phrase literally by suggesting that the cricket team brought a ladder to reach these "high" scores. The humor comes from the absurdity of needing a ladder to achieve something that is metaphorical rather than physical. It\'s a clever twist that combines sports terminology with a playful interpretation of language.'}

In [11]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'cricket', 'joke': 'Why did the cricket team bring a ladder to the match?\n\nBecause they heard the game was going to be a "high" scoring one!', 'explanation': 'This joke plays on a pun involving the word "high." In cricket, a "high-scoring" game refers to a match where teams score a lot of runs, indicating an exciting and competitive game. However, the joke takes this phrase literally by suggesting that the cricket team brought a ladder to reach these "high" scores. The humor comes from the absurdity of needing a ladder to achieve something that is metaphorical rather than physical. It\'s a clever twist that combines sports terminology with a playful interpretation of language.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1e27a6-975e-6a47-8002-26bf6653393f'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-11-17T11:31:20.841866+00:00', parent_config={'configurable': {'thread_id'